# Uber Data Analysis (Nov 2024) - Rookie B.Tech Notebook

Put your dataset in `../data/uber_dataset.csv` (or change `DATA_PATH` below).

This notebook creates visuals for:
- Demand trends and peak hours
- Surge/pricing patterns (if the dataset has a surge-like column)
- High-traffic geo hotspots
- Correlation heatmap
- Time-series demand plot


In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from src.uber_eda import (
    load_data,
    clean_data,
    add_geo_bins,
    plot_peak_hours,
    plot_demand_time_series,
    plot_geo_hotspots,
    plot_correlation_heatmap,
    plot_surge_or_pricing_patterns,
)

DATA_PATH = os.path.join(PROJECT_ROOT, "data", "uber_dataset.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")

df = load_data(DATA_PATH)
print("Loaded shape:", df.shape)
print("Columns:", list(df.columns)[:15], "...")


In [ ]:
df_clean, meta = clean_data(df)
print("Detected datetime column:", meta.datetime_col)
print("Detected lat/lon:", meta.lat_col, meta.lon_col)
print("Detected surge/pricing signal:", meta.surge_col)
print("Cleaned shape:", df_clean.shape)

# Rookie-friendly demand peak-hour insight using groupby
demand_by_hour = df_clean.groupby("hour").size().sort_index()
peak_hour = int(demand_by_hour.idxmax())
print("Peak hour:", peak_hour)
print("Rides at peak hour:", int(demand_by_hour.loc[peak_hour]))


In [ ]:
# 1) Peak hours plot
peak_info = plot_peak_hours(df_clean, OUTPUT_DIR)
peak_info


In [ ]:
# 2) Time-series demand plot
plot_demand_time_series(df_clean, OUTPUT_DIR)

# 3) Geo hotspots plot (if lat/lon exists)
if meta.lat_col and meta.lon_col:
    binned = add_geo_bins(df_clean, meta.lat_col, meta.lon_col, precision=2)
    top_geo = plot_geo_hotspots(binned, OUTPUT_DIR, top_n=10, precision=2)
    print("Top geo hotspots (top bins):")
    top_geo[:5]
else:
    print("No lat/lon columns detected; skipping geo hotspots.")


In [ ]:
# 4) Correlation heatmap (numeric features)
plot_correlation_heatmap(df_clean, OUTPUT_DIR)

# 5) Surge/pricing patterns heatmap (if a surge-like column exists)
surge_summary = plot_surge_or_pricing_patterns(df_clean, OUTPUT_DIR, surge_col=meta.surge_col)
surge_summary

print("Done. Check generated images in:", OUTPUT_DIR)
